![Practicum AI Logo image](https://github.com/PracticumAI/practicumai.github.io/blob/main/images/logo/PracticumAI_logo_250x50.png?raw=true) <img src="https://github.com/PracticumAI/practicumai.github.io/blob/84b04be083ca02e5c7e92850f9afd391fc48ae2a/images/icons/practicumai_computer_vision.png?raw=true" alt="Practicum AI: Computer Vision icon" align="right" width=50>
***

# Transfer Learning Concepts

You may recall *Practicum AI*"s heroine Amelia, the AI-savvy nutritionist. At the end of our *[Deep Learning Foundations course](https://practicumai.org/courses/deep_learning/)*, Amelia was helping with a computer vision project. If only she had known about transfer learning, it could have saved her a lot of time! In this notebook, we will get some hands-on experience with transfer learning and show you how to use it to improve your workflows.

![Figure 2 of the AgriNet paper used as the cover image for this notebook. Figure 2 depicts using transfer learning to make a computer vision model more efficient](images/agrinet_figure-cover.jpg)


## AI Pathway review for Transfer Learning & AgriNet 

If you have taken our [Getting Started with AI course](https://practicumai.org/courses/getting_started/), you may remember this figure of the AI Application Development Pathway. Let's take a quick review of how we will apply this to our case study of AgriNet and it's use of transfer learning.

![AI Application Development Pathway image showing the 7 steps in developing an AI application](https://practicumai.org/getting_started/images/application_dev_pathway.png)

1. **Choose a problem to solve:** In this example, we will be trying to make a computer vision model that can recognize images of plants. 
2. **Gather data:** The data for the example comes from [HuggingFace](https://www.huggingface.co//), a great repository of datasets, code, and models.
3. **Clean and prepare the data:** In the *Deep Learning Foundations* course, we assumed that this was done for us. One issue that we ran into was that of class imbalance. Here, the (probably very tired) researchers that created the AgriNet dataset have already balanced the classes for us!
4. **Choose a model:** In the *Deep Learning Foundations* course, we presented the model with little detail. Here, we will be using a pre-trained model, ResNet50, and applying transfer learning to it.
   * In the step where you'd choose a model, one can approach this in two ways:
      * **Train from scratch:** This is where you start with a randomly initialized model and train it on your data. This can be computationally expensive and time-consuming.
      * **Transfer learning:** This is where you start with a pre-trained model and fine-tune it on your data. This is often faster and requires less data.
5. **Train the model:** We'll be comparing training a model from scratch to transfering to a pre-trained model. With so many hyperparameters to tune and compare, it's easy to lose track of what combinations have been tried and how changes impacted model performance. 
   * In this notebook, we introduce you to [TensorBoard](https://www.tensorflow.org/tensorboard), one popular tool in a class of tools known as **experiment tracking** or **MLOps (Machine learning operations) tools**. These tools help track changes to hyperparameters, the training process, and the data. They allow comparison among runs and can even automate multiple runs for you. Learning to use MLOps tools will help you as you continue to learn more about AI workflows.
   * We'll demonstrate three approaches in this notebook:
      - Training a baseline model from scratch.
      - Fine-tuning a model pre-trained on ImageNet.
      - Fine-tuning a model pre-trained on AgriNet, a domain-specific dataset.
6. **Evaluate the model:** We will use the metrics we gather to make decisions about the model. 
7. **Deploy the model:** We won't get to this stage in this exercise, but hopefully, we will end up with a model that could be deployed and achieve relatively good accuracy at solving the problem.


### A Refresher

If you need a refresher, or havent taken the *Deep Learning Foundations* course, the final notebook is part of this repository: [DLF_03_bees_vs_wasps.ipynb](DLF_03_bees_vs_wasps.ipynb).

### A Quick Primer on the Baseline Model
We'll train a simple convolutional neural network (CNN) from scratch as a baseline for comparison.

We'll define a CNN with basic layers, such as convolutional, pooling, and fully connected layers. The model will be compiled with the Adam optimizer and categorical cross-entropy loss, then trained on the dataset. Strictly speaking, a thorough knowledge of CNNs is not required for this notebook, but if you're interested in learning more, we recommend the our [PracticumAI: Computer Vision](https://github.com/PracticumAI/computer_vision) Intermediate course.

That said, with *any* machine learning work, the better you understand the model, the better you can tune it to your needs.


### Transfer Learning with ImageNet

We'll use the VGG19 model pre-trained on ImageNet and fine-tune it for plant disease detection.

ImageNet pre-trained models have learned general features (e.g., edges, textures) that can be adapted to our specific task. This significantly reduces the training time and data requirements.

The base layers of VGG19 will be frozen to retain their pre-trained features. We'll add custom layers for classification and fine-tune the model on our dataset.

## 1. Import the libraries we will use

In [2]:
%pip install torch

Defaulting to user installation because normal site-packages is not writeable
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/204.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/204.1 MB 11.7 MB/s eta 0:00:18
    --------------------------------------- 4.7/204.1 MB 11.9 MB/s eta 0:00:17
   - -------------------------------------- 7.1/204.1 MB 11.8 MB/s eta 0:00:17
   - -------------------------------------- 9.7/204.1 MB 11.6 MB/s eta 0:00:17
   -- ------------------------------------- 12.1/204.1 MB 11.6 MB/s eta 0:00:17
   -- ------------------------------------- 14.4/204.1 MB 11.6 MB/s eta 0:00:17
   --- ------------------------------------ 16.8/204.1 MB 11.6 MB/s eta 0:00:17
   --- ------------------------------------ 19.1/204.1 MB 11.6 MB/s eta 0:00:16
   ---- ----------------------------------- 21.8/204.1 MB 11.7 MB/s eta 0:00:16
   ---- ----------------------------------- 24.1/204.1 MB 11.7 MB/s eta 


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: C:\Users\sushi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
%pip install torchvision

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 1.6/1.6 MB 9.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: C:\Users\sushi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [5]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from PIL import ImageFile
import requests
import zipfile
import matplotlib.pyplot as plt

## 2. Jupyter magic commands

In Jupyter, the `%` is used as a "magic" command. These extend Python in various ways. In this case, Tensorboad functionality is added using `%load_ext tensorboard`

Again, [Tensorboard](https://www.tensorflow.org/tensorboard) is the tool we"ll use for experiment tracking in many of these notebooks. 

In [6]:
# Load the TensorBoard notebook extension
%load_ext tensorboard

## 3. Getting the data

Gotta have data to train a model! The code below downloads the curated version of the dataset and unzips it.

In [8]:
# Download the dataset, extract it to the data folder and remove the zip file
download_path = "https://data.rc.ufl.edu/pub/practicum-ai/Transfer_Learning_Intermediate/agrinet_curated.zip"
zip_path = "data/agrinet_curated.zip"
data_path = "data"

# Paths to dataset
train_dir = os.path.join(data_path, "agri_net_train")
val_dir = os.path.join(data_path, "agri_net_val")
test_dir = os.path.join(data_path, "agri_net_test")

# Check if the data is already loaded
if not (os.path.exists(train_dir) and os.path.exists(val_dir) and os.path.exists(test_dir)):
    # Create the data directory if it does not exist
    if not os.path.exists(data_path):
        os.makedirs(data_path)

    # Download the zip file
    r = requests.get(download_path)
    with open(zip_path, "wb") as f:
        f.write(r.content)

    # Extract the zip file
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(data_path)

    # Remove the zip file
    os.remove(zip_path)
else:
    print("Data is already loaded.")

Data is already loaded.


## 4. Preparing the generators

Data generators are used to load data in batches and preprocess it. We'll use the `ImageDataGenerator` class from Keras to load and preprocess the data.

In [ ]:
# Define PyTorch data transforms
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
}

# Load PyTorch datasets
image_datasets = {
    'train': datasets.ImageFolder(train_dir, data_transforms['train']),
    'val': datasets.ImageFolder(val_dir, data_transforms['val'])
}

# Create PyTorch data loaders
dataloaders = {
    'train': torch.utils.data.DataLoader(image_datasets['train'], batch_size=32, shuffle=True),
    'val': torch.utils.data.DataLoader(image_datasets['val'], batch_size=32, shuffle=False)
}

### Transfer Learning with AgriNet

We'll use the VGG19 model pre-trained on the AgriNet dataset, which is domain-specific to agriculture. Domain-specific pre-training captures features relevant to agricultural tasks, such as plant patterns and disease characteristics, which can further improve model performance compared to generic pre-trained models. Similar to the ImageNet approach, we'll freeze the base layers of the AgriNet model, add custom classification layers, and fine-tune the model on our dataset.

## Baseline Model

We'll train a simple convolutional neural network from scratch and use it as our baseline for performance comparison.

### Performance Comparison

We'll compare the performance of the three models (baseline, ImageNet pre-trained, and AgriNet pre-trained) using metrics like accuracy and F1-score. This step helps quantify the benefits of transfer learning and highlights the impact of using domain-specific pre-trained models. We'll evaluate each model on the test set and visualize the results using performance metrics and charts.

In [ ]:
# Handle truncated images
ImageFile.LOAD_TRUNCATED_IMAGES = True

# Define baseline model using PyTorch
class BaselineModel(nn.Module):
    def __init__(self, num_classes):
        super(BaselineModel, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 112 * 112, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

num_classes = len(image_datasets['train'].classes)
baseline_model_pt = BaselineModel(num_classes)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(baseline_model_pt.parameters(), lr=0.001)

# Train the baseline model using PyTorch
num_epochs = 10
for epoch in range(num_epochs):
    baseline_model_pt.train()
    running_loss = 0.0
    for inputs, labels in dataloaders['train']:
        optimizer.zero_grad()
        outputs = baseline_model_pt(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
    epoch_loss = running_loss / len(image_datasets['train'])
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}')

### Conclusion: Key Insights

- Transfer learning significantly improves performance compared to training from scratch, especially with limited data.
- Domain-specific pre-training (e.g., AgriNet) can further enhance accuracy and generalization for specialized tasks.
- These findings demonstrate the importance of transfer learning in tackling real-world challenges in agriculture.

## Transfer Learning with ImageNet

We'll use a pre-trained VGG19 model with ImageNet weights and fine-tune it on our dataset.

In [ ]:
# Load pre-trained VGG19 model using PyTorch
imagenet_model_pt = models.vgg19(pretrained=True)

# Freeze base layers
for param in imagenet_model_pt.parameters():
    param.requires_grad = False

# Add custom top layers
num_ftrs = imagenet_model_pt.classifier[0].in_features
imagenet_model_pt.classifier = nn.Sequential(
    nn.Linear(num_ftrs, 128),
    nn.ReLU(inplace=True),
    nn.Dropout(0.5),
    nn.Linear(128, num_classes)
)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(imagenet_model_pt.classifier.parameters(), lr=0.001)

# Train the model using PyTorch
num_epochs = 10
for epoch in range(num_epochs):
    imagenet_model_pt.train()
    running_loss = 0.0
    for inputs, labels in dataloaders['train']:
        optimizer.zero_grad()
        outputs = imagenet_model_pt(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
    epoch_loss = running_loss / len(image_datasets['train'])
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}')

In [ ]:
# Assuming AgriNet weights are available locally
agri_weights_path = "/path/to/agri_vgg19_weights.pth"  # Replace with actual path

# Load the VGG19 model using PyTorch
agri_model_pt = models.vgg19(pretrained=False)

# Load AgriNet weights
agri_model_pt.load_state_dict(torch.load(agri_weights_path))

# Freeze base layers
for param in agri_model_pt.parameters():
    param.requires_grad = False

# Add custom top layers
num_ftrs = agri_model_pt.classifier[0].in_features
agri_model_pt.classifier = nn.Sequential(
    nn.Linear(num_ftrs, 128),
    nn.ReLU(inplace=True),
    nn.Dropout(0.5),
    nn.Linear(128, num_classes)
)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(agri_model_pt.classifier.parameters(), lr=0.001)

# Train the model using PyTorch
num_epochs = 10
for epoch in range(num_epochs):
    agri_model_pt.train()
    running_loss = 0.0
    for inputs, labels in dataloaders['train']:
        optimizer.zero_grad()
        outputs = agri_model_pt(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
    epoch_loss = running_loss / len(image_datasets['train'])
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}')

## Conclusion

In this notebook, we demonstrated the benefits of transfer learning in agricultural tasks. The AgriNet pre-trained model outperformed the ImageNet model and the baseline, showing the importance of domain-specific pre-training for specialized applications.